In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

In [3]:
# PALETTE & THEME
BG      = '#0D0D0D'
PAPER   = '#141414'
GRID    = '#2a2a2a'
TEXT    = '#E8E8E8'
SUBTEXT = '#999999'
ACCENT  = '#FF6B35'

# Helper: convert hex color to rgba string with given alpha
def hex_to_rgba(hex_color, alpha=0.33):
    h = hex_color.lstrip('#')
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f'rgba({r},{g},{b},{alpha})'

SEG_COLORS = {
    'Commuter (<125cc)'    : '#38BDF8',
    'Mid-Range (125-300cc)': '#A78BFA',
    'Premium (300-600cc)'  : '#FB923C',
    'Super Bike (600cc+)'  : '#F43F5E',
}
TYPE_COLORS = {'Petrol': '#FF6B35', 'Electric': '#00D4FF'}

def base_layout(title='', h=520):
    return dict(
        title=dict(text=title, font=dict(size=18, color=TEXT, family='Poppins, sans-serif'),
                   x=0.03, y=0.97),
        height=h,
        paper_bgcolor=PAPER,
        plot_bgcolor=BG,
        font=dict(color=TEXT, family='Poppins, sans-serif', size=12),
        margin=dict(l=60, r=30, t=70, b=60),
        xaxis=dict(gridcolor=GRID, zerolinecolor=GRID, tickfont=dict(color=SUBTEXT)),
        yaxis=dict(gridcolor=GRID, zerolinecolor=GRID, tickfont=dict(color=SUBTEXT)),
        legend=dict(bgcolor='rgba(0,0,0,0)', bordercolor=GRID,
                    font=dict(color=TEXT, size=11)),
    )

all_figs = []

In [4]:
# Load CSV (already cleaned — all base columns pre-computed)
df = pd.read_csv('Scrapping_cleaned.csv')

# Derive Brand (not in CSV)
def get_brand(model):
    words = str(model).split()
    if len(words) > 1 and words[0] == 'Royal' and words[1] == 'Enfield':
        return 'Royal Enfield'
    return words[0]
df['Brand'] = df['Model Name'].apply(get_brand)

# Derive Engine Segment for petrol bikes (not in CSV)
def eng_seg(cc):
    if cc < 125:  return 'Commuter (<125cc)'
    if cc < 300:  return 'Mid-Range (125-300cc)'
    if cc < 600:  return 'Premium (300-600cc)'
    return 'Super Bike (600cc+)'

petrol_mask = df['Bike_Type'] == 'Petrol'
df.loc[petrol_mask, 'Engine_Seg'] = df.loc[petrol_mask, 'Engine_val'].apply(eng_seg)

print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head(3)

Shape: (418, 12)
Columns: ['Model Name', 'Engine Size', 'Average Rating', 'Number of Ratings', 'Price', 'Bike_Type', 'Engine_val', 'Mileage_val', 'Mileage_unit', 'Price_Category', 'Brand', 'Engine_Seg']


,Model Name,Engine Size,Average Rating,Number of Ratings,Price,Bike_Type,Engine_val,Mileage_val,Mileage_unit,Price_Category,Brand,Engine_Seg
0,KTM 200 Duke,199.5 cc,4.7,621.0,191570.0,Petrol,199.50,35.0,kmpl,Mid-Range (₹1–3L),KTM,Mid-Range (125-300cc)
1,Bajaj Pulsar N250,249 cc,4.6,398.0,138823.0,Petrol,249.00,44.0,kmpl,Mid-Range (₹1–3L),Bajaj,Mid-Range (125-300cc)
2,Honda CB350RS,348.36 cc,4.6,223.0,197914.0,Petrol,348.36,35.0,kmpl,Mid-Range (₹1–3L),Honda,Premium (300-600cc)


### 1. PRICE DISTRIBUTION — Histogram

In [5]:
fig1 = go.Figure()

fig1.add_trace(go.Histogram(
    x=df['Price'],
    nbinsx=50,
    marker=dict(
        color='#00C2FF',
        line=dict(
            color='white',
            width=0.5
        )
    ),
    opacity=0.9,
    hovertemplate='Price: %{x:,.0f}<br>Count: %{y}<extra></extra>'
))

fig1.update_layout(**base_layout('Price Distribution of All Bikes'))

fig1.show()

### 2. BRAND DISTRIBUTION — Bar Chart

In [6]:
brand_counts = df['Brand'].value_counts().reset_index()
brand_counts.columns = ['Brand', 'Count']

fig2 = go.Figure(go.Bar(
    x=brand_counts['Brand'],
    y=brand_counts['Count'],
    text=brand_counts['Count'],
    textposition='outside',
    marker=dict(
        color=brand_counts['Count'],
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title='Models', tickfont=dict(color=TEXT)),
        line=dict(color=BG, width=0.5),
    ),
    hovertemplate='<b>%{x}</b><br>Models: %{y}<extra></extra>',
))
fig2.update_layout(**base_layout('Number of Models by Brand'), showlegend=False)
fig2.update_xaxes(title='Brand', tickangle=-45)
fig2.update_yaxes(title='Number of Models')
all_figs.append(('2. Brand Distribution', fig2))
fig2.show()

### 3. PETROL vs ELECTRIC — Donut Chart

In [7]:
type_counts = df['Bike_Type'].value_counts().reset_index()
type_counts.columns = ['Bike_Type', 'Count']

fig3 = go.Figure(go.Pie(
    labels=type_counts['Bike_Type'],
    values=type_counts['Count'],
    hole=0.45,
    marker=dict(
        colors=[TYPE_COLORS.get(t, ACCENT) for t in type_counts['Bike_Type']],
        line=dict(color=PAPER, width=2),
    ),
    textinfo='label+percent',
    hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Share: %{percent}<extra></extra>',
))
fig3.update_layout(**base_layout('Petrol vs Electric Bikes'))
all_figs.append(('3. Petrol vs Electric', fig3))
fig3.show()

### 4. PRICE CATEGORY — Donut Chart

In [8]:
cat_counts = df['Price_Category'].value_counts().reset_index()
cat_counts.columns = ['Price_Category', 'Count']
palette = ['#34D399', '#60A5FA', '#FBBF24', '#F87171']

fig4 = go.Figure(go.Pie(
    labels=cat_counts['Price_Category'],
    values=cat_counts['Count'],
    hole=0.5,
    marker=dict(
        colors=palette[:len(cat_counts)],
        line=dict(color=PAPER, width=2),
    ),
    textinfo='label+percent',
    hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Share: %{percent}<extra></extra>',
))
fig4.update_layout(**base_layout('Price Category Breakdown'))
all_figs.append(('4. Price Categories', fig4))
fig4.show()

### 5. ENGINE SIZE DISTRIBUTION — Violin Plot (Petrol only)

In [9]:
petrol_df = df[df['Bike_Type'] == 'Petrol'].dropna(subset=['Engine_val'])

fig5 = go.Figure(go.Violin(
    y=petrol_df['Engine_val'],
    box_visible=True,
    meanline_visible=True,
    fillcolor='#A78BFA',
    line_color='#E8E8E8',
    opacity=0.75,
    name='Engine Size (cc)',
    hovertemplate='Engine Size: %{y:.1f} cc<extra></extra>',
))
fig5.update_layout(**base_layout('Engine Size Distribution (Petrol Bikes)'))
fig5.update_yaxes(title='Engine Size (cc)')
all_figs.append(('5. Engine Size Violin', fig5))
fig5.show()

### 6. PRICE DISTRIBUTION — Violin Plot

In [10]:
fig6 = go.Figure(go.Violin(
    y=df['Price'],
    box_visible=True,
    meanline_visible=True,
    fillcolor='#EF553B',
    line_color='#E8E8E8',
    opacity=0.75,
    name='Price',
    hovertemplate='Price: %{y:,.0f}<extra></extra>',
))
fig6.update_layout(**base_layout('Price Distribution - Violin Plot'))
fig6.update_yaxes(title='Price (Rs)')
all_figs.append(('6. Price Violin', fig6))
fig6.show()

### 7. PRICE vs ENGINE SIZE — Scatter Plot

In [11]:
sdf = df.dropna(subset=['Engine_val', 'Price']).copy()

fig7 = go.Figure()
for btype, color in TYPE_COLORS.items():
    sub = sdf[sdf['Bike_Type'] == btype]
    if sub.empty:
        continue
    fig7.add_trace(go.Scatter(
        x=sub['Engine_val'],
        y=sub['Price'],
        mode='markers',
        name=btype,
        marker=dict(color=color, size=7, opacity=0.75,
                    line=dict(color=BG, width=0.5)),
        hovertemplate=(
            '<b>%{text}</b><br>'
            'Engine: %{x:.1f}<br>'
            'Price: %{y:,.0f}<extra></extra>'
        ),
        text=sub['Model Name'],
    ))
fig7.update_layout(**base_layout('Price vs Engine Size'))
fig7.update_xaxes(title='Engine Size (cc / kWh)')
fig7.update_yaxes(title='Price (Rs)')
all_figs.append(('7. Price vs Engine', fig7))
fig7.show()

### 8. MILEAGE vs PRICE — Scatter Plot (Petrol, kmpl only)

In [12]:
mdf = df[
    (df['Bike_Type'] == 'Petrol') &
    (df['Mileage_unit'] == 'kmpl')
].dropna(subset=['Mileage_val', 'Price']).copy()

fig8 = go.Figure(go.Scatter(
    x=mdf['Price'],
    y=mdf['Mileage_val'],
    mode='markers',
    marker=dict(
        color=mdf['Price'],
        colorscale='Plasma',
        showscale=True,
        colorbar=dict(title='Price', tickfont=dict(color=TEXT)),
        size=8,
        opacity=0.8,
        line=dict(color=BG, width=0.4),
    ),
    hovertemplate=(
        '<b>%{text}</b><br>'
        'Price: %{x:,.0f}<br>'
        'Mileage: %{y:.1f} kmpl<extra></extra>'
    ),
    text=mdf['Model Name'],
))
fig8.update_layout(**base_layout('Mileage vs Price (Petrol Bikes, kmpl)'))
fig8.update_xaxes(title='Price (Rs)',
                  tickvals=[1e5, 3e5, 6e5, 1e6, 3e6],
                  ticktext=['1L', '3L', '6L', '10L', '30L'])
fig8.update_yaxes(title='Mileage (kmpl)')
all_figs.append(('8. Mileage vs Price', fig8))
fig8.show()

### 9. AVERAGE RATING DISTRIBUTION — Histogram

In [13]:
rdf = df.dropna(subset=['Average Rating']).copy()

fig9 = go.Figure(go.Histogram(
    x=rdf['Average Rating'],
    nbinsx=30,
    marker=dict(
        color='#34D399',
        line=dict(
            color='white',
            width=0.5
        )
    ),
    opacity=0.9,
    hovertemplate='Rating: %{x:.1f}<br>Count: %{y}<extra></extra>'
))

fig9.update_layout(**base_layout('Average Rating Distribution'))

fig9.update_xaxes(title='Average Rating')
fig9.update_yaxes(title='Number of Bikes')

fig9.show()

### 10. PRICE by ENGINE SEGMENT — Box Plot

In [14]:
# FIX: fillcolor must use rgba() format — 8-digit hex (#RRGGBBAA) is NOT supported by Plotly
seg_order = [
    'Commuter (<125cc)',
    'Mid-Range (125-300cc)',
    'Premium (300-600cc)',
    'Super Bike (600cc+)',
]
seg_df = df.dropna(subset=['Engine_Seg', 'Price'])

fig10 = go.Figure()
for seg in seg_order:
    sub = seg_df[seg_df['Engine_Seg'] == seg]
    if sub.empty:
        continue
    hex_c = SEG_COLORS.get(seg, ACCENT)
    fig10.add_trace(go.Box(
        y=sub['Price'],
        name=seg,
        marker_color=hex_c,
        line_color=hex_c,
        fillcolor=hex_to_rgba(hex_c, 0.33),   # rgba() instead of 8-digit hex
        hovertemplate='<b>' + seg + '</b><br>Price: %{y:,.0f}<extra></extra>',
    ))
fig10.update_layout(**base_layout('Price Distribution by Engine Segment'))
fig10.update_yaxes(title='Price (Rs)')
fig10.update_xaxes(title='Engine Segment')
all_figs.append(('10. Price by Segment Box', fig10))
fig10.show()

### 11. TOP 10 BRANDS — Average Price

In [16]:
brand_price = (
    df.groupby('Brand')['Price']
      .agg(['mean', 'count'])
      .reset_index()
      .rename(columns={'mean': 'Avg_Price', 'count': 'Count'})
)
brand_price = (
    brand_price[brand_price['Count'] >= 3]
    .sort_values('Avg_Price', ascending=False)
    .head(10)
)

fig11 = go.Figure(go.Bar(
    x=brand_price['Brand'],
    y=brand_price['Avg_Price'],
    text=brand_price['Avg_Price'].apply(lambda x: f'{x/1e5:.1f}L'),
    textposition='outside',
    marker=dict(
        color=brand_price['Avg_Price'],
        colorscale='Turbo',
        showscale=True,
        colorbar=dict(
            title='Avg Price',
            tickfont=dict(color=TEXT)
        ),
        line=dict(color='white', width=0.5)
    ),
    hovertemplate='<b>%{x}</b><br>Avg Price: %{y:,.0f}<extra></extra>',
))

fig11.update_layout(
    **base_layout('Top 10 Brands by Average Price'),
    showlegend=False
)

fig11.update_xaxes(title='Brand', tickangle=-30)
fig11.update_yaxes(title='Average Price (Rs)')

fig11.show()

### 12. RATING vs PRICE — Bubble Chart

In [ ]:
bdf = df.dropna(subset=['Average Rating', 'Price', 'Number of Ratings']).copy()
max_ratings = bdf['Number of Ratings'].max()

fig12 = go.Figure(go.Scatter(
    x=bdf['Average Rating'],
    y=bdf['Price'],
    mode='markers',
    marker=dict(
        size=np.clip(bdf['Number of Ratings'] / max_ratings * 40, 5, 40),
        color=bdf['Price'],
        colorscale='Plasma',
        showscale=True,
        colorbar=dict(title='Price', tickfont=dict(color=TEXT)),
        opacity=0.75,
        line=dict(color=BG, width=0.4),
    ),
    hovertemplate=(
        '<b>%{text}</b><br>'
        'Rating: %{x:.1f}<br>'
        'Price: %{y:,.0f}<extra></extra>'
    ),
    text=bdf['Model Name'],
))
fig12.update_layout(**base_layout('Rating vs Price Bubble Chart (bubble size = num ratings)'))
fig12.update_xaxes(title='Average Rating')
fig12.update_yaxes(title='Price (Rs)')
all_figs.append(('12. Rating vs Price Bubble', fig12))
fig12.show()

### 13. CUMULATIVE DISTRIBUTION (CDF) — Price

In [ ]:
sorted_prices = np.sort(df['Price'].dropna().values)
cdf = np.arange(1, len(sorted_prices) + 1) / len(sorted_prices)

fig13 = go.Figure(go.Scatter(
    x=sorted_prices,
    y=cdf,
    mode='lines',
    line=dict(color=ACCENT, width=2.5),
    fill='tozeroy',
    fillcolor='rgba(255,107,53,0.15)',
    hovertemplate='Price: %{x:,.0f}<br>CDF: %{y:.2%}<extra></extra>',
    name='CDF',
))
fig13.update_layout(**base_layout('Cumulative Distribution (CDF) of Bike Prices'))
fig13.update_xaxes(title='Price (Rs)',
                   tickvals=[1e5, 3e5, 6e5, 1e6, 3e6, 6e6, 1e7],
                   ticktext=['1L', '3L', '6L', '10L', '30L', '60L', '1Cr'])
fig13.update_yaxes(title='Proportion of Bikes (0-1)', tickformat='.0%')
all_figs.append(('13. CDF Price', fig13))
fig13.show()

### 14. SUMMARY DASHBOARD — 4-panel overview

In [ ]:
fig14 = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Price Distribution',
        'Brand Model Count (Top 12)',
        'Petrol vs Electric Share',
        'Rating Distribution',
    ],
    specs=[
        [{'type': 'histogram'}, {'type': 'bar'}],
        [{'type': 'pie'},       {'type': 'histogram'}],
    ],
)

# Panel 1 (row=1, col=1): Price histogram
fig14.add_trace(go.Histogram(
    x=df['Price'], nbinsx=40,
    marker_color=ACCENT, opacity=0.85, showlegend=False,
    hovertemplate='%{x:,.0f}<br>Count: %{y}<extra></extra>',
), row=1, col=1)

# Panel 2 (row=1, col=2): Top 12 brands
top12 = df['Brand'].value_counts().head(12).reset_index()
top12.columns = ['Brand', 'Count']
fig14.add_trace(go.Bar(
    x=top12['Brand'], y=top12['Count'],
    marker_color='#38BDF8', showlegend=False,
    hovertemplate='<b>%{x}</b><br>%{y} models<extra></extra>',
), row=1, col=2)

# Panel 3 (row=2, col=1): Petrol vs Electric pie
tc = df['Bike_Type'].value_counts()
fig14.add_trace(go.Pie(
    labels=tc.index.tolist(),
    values=tc.values.tolist(),
    hole=0.4,
    marker=dict(
        colors=[TYPE_COLORS.get(t, ACCENT) for t in tc.index],
        line=dict(color=PAPER, width=2),
    ),
    showlegend=True,
    hovertemplate='<b>%{label}</b><br>%{value} bikes (%{percent})<extra></extra>',
), row=2, col=1)

# Panel 4 (row=2, col=2): Rating histogram
rdf2 = df.dropna(subset=['Average Rating'])
fig14.add_trace(go.Histogram(
    x=rdf2['Average Rating'], nbinsx=25,
    marker_color='#A78BFA', opacity=0.85, showlegend=False,
    hovertemplate='Rating: %{x:.1f}<br>Count: %{y}<extra></extra>',
), row=2, col=2)

fig14.update_layout(
    height=700,
    paper_bgcolor=PAPER,
    plot_bgcolor=BG,
    font=dict(color=TEXT, family='Poppins, sans-serif', size=11),
    title=dict(text='Summary Dashboard - BikeWale Dataset',
               font=dict(size=18, color=TEXT), x=0.03),
    margin=dict(l=50, r=30, t=80, b=50),
)

# Update subplot annotation text colour
for ann in fig14.layout.annotations:
    ann.font.color = TEXT
    ann.font.size  = 12

# FIX: row=2,col=2 histogram uses xaxis3/yaxis3 (not xaxis4/yaxis4)
# Apply dark grid only to axes that actually exist
for attr in ['xaxis', 'yaxis', 'xaxis2', 'yaxis2', 'xaxis3', 'yaxis3']:
    ax = getattr(fig14.layout, attr, None)
    if ax is not None:
        ax.gridcolor     = GRID
        ax.zerolinecolor = GRID

all_figs.append(('14. Summary Dashboard', fig14))
fig14.show()

### Summary — all visualizations

In [ ]:
print(f'Total visualizations created: {len(all_figs)}')
for i, (title, _) in enumerate(all_figs, 1):
    print(f'  {i:2d}. {title}')

Total visualizations created: 14
   1. 1. Price Distribution
   2. 2. Brand Distribution
   3. 3. Petrol vs Electric
   4. 4. Price Categories
   5. 5. Engine Size Violin
   6. 6. Price Violin
   7. 7. Price vs Engine
   8. 8. Mileage vs Price
   9. 9. Rating Distribution
  10. 10. Price by Segment Box
  11. 11. Brand Avg Price
  12. 12. Rating vs Price Bubble
  13. 13. CDF Price
  14. 14. Summary Dashboard
